# 01 - Metadata Merge, Duplicate Audit, and Image Quality Audit

**Pipeline:** Skin Cancer 3-Class Classification (NV / MEL / BCC)
**Input:** `pipe output/01_reconciliation/01_reconciled_manifest.csv`
**Purpose:** Merge ISIC metadata, detect duplicate/family risks, decode images, compute quality metrics, and build one combined manual review queue.

**Out of scope for this notebook:** row exclusion, preprocessing, augmentation, splitting, training.

---

## Section 0 - Imports

In [1]:
import hashlib
import os
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

CV2_AVAILABLE = False
PIL_AVAILABLE = False

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    pass

if not CV2_AVAILABLE:
    try:
        from PIL import Image
        PIL_AVAILABLE = True
    except ImportError:
        pass

if not CV2_AVAILABLE and not PIL_AVAILABLE:
    warnings.warn("Neither cv2 nor PIL found. Image decoding will be skipped.")

print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print(f"cv2    : {CV2_AVAILABLE}")
print(f"PIL    : {PIL_AVAILABLE}")

pandas : 2.3.3
numpy  : 2.4.4
cv2    : True
PIL    : False


## Section 1 - Paths and Configuration

In [2]:
RECONCILED_MANIFEST = Path(r"C:\SKIN CANCER v2\pipe output\01_reconciliation\01_reconciled_manifest.csv")
METADATA_CSV        = Path(r"C:\SKIN CANCER v2\DS\ISIC_2019_Training_Metadata.csv")
OUTPUT_AUDIT_DIR    = Path(r"C:\SKIN CANCER v2\pipe output\02_audit_review")

MIN_DIM_PX          = 64
EXTREME_AR_HIGH     = 4.0
EXTREME_AR_LOW      = 0.25
DARK_THRESH         = 15.0
BRIGHT_THRESH       = 245.0
LOW_CONTRAST_THRESH = 5.0
LOW_SHARP_THRESH    = 5.0

print("Paths configured.")
print(f"  Input manifest : {RECONCILED_MANIFEST}")
print(f"  Metadata CSV   : {METADATA_CSV}")
print(f"  Output dir     : {OUTPUT_AUDIT_DIR}")

Paths configured.
  Input manifest : C:\SKIN CANCER v2\pipe output\01_reconciliation\01_reconciled_manifest.csv
  Metadata CSV   : C:\SKIN CANCER v2\DS\ISIC_2019_Training_Metadata.csv
  Output dir     : C:\SKIN CANCER v2\pipe output\02_audit_review


## Section 2 - Create Output Folder

In [3]:
OUTPUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {OUTPUT_AUDIT_DIR}")

Ready: C:\SKIN CANCER v2\pipe output\02_audit_review


## Section 3 - Load Reconciled Manifest

In [4]:
df = pd.read_csv(RECONCILED_MANIFEST, low_memory=False)
print(f"Loaded: {len(df):,} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

elig_mask = df["eligible_after_reconciliation"] == True
n_elig    = elig_mask.sum()
print(f"\nReconciliation-eligible rows : {n_elig:,}")
print(f"Ineligible rows              : {(~elig_mask).sum():,}")
df.head(3)

Loaded: 20,720 rows x 18 columns
Columns: ['full_path', 'folder_name', 'file_name_with_extension', 'file_stem_raw', 'extension', 'recognized_suffix', 'base_id_candidate', 'canonical_match_id', 'folder_label', 'gt_label', 'final_authoritative_label', 'class_index', 'match_status', 'label_agreement_status', 'file_exists', 'file_size_bytes', 'eligible_after_reconciliation', 'reconciliation_issue_reason']

Reconciliation-eligible rows : 20,720
Ineligible rows              : 0


,full_path,folder_name,file_name_with_extension,file_stem_raw,extension,recognized_suffix,base_id_candidate,canonical_match_id,folder_label,gt_label,final_authoritative_label,class_index,match_status,label_agreement_status,file_exists,file_size_bytes,eligible_after_reconciliation,reconciliation_issue_reason
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,ISIC_0000000.jpg,ISIC_0000000,.jpg,NaN,ISIC_0000000,ISIC_0000000,NV,NV,NV,0,matched_exact,agree,True,49964,True,NaN
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,ISIC_0000001.jpg,ISIC_0000001,.jpg,NaN,ISIC_0000001,ISIC_0000001,NV,NV,NV,0,matched_exact,agree,True,38941,True,NaN
2,C:\SKIN CANCER v2\DS\NV\ISIC_0000003.jpg,NV,ISIC_0000003.jpg,ISIC_0000003,.jpg,NaN,ISIC_0000003,ISIC_0000003,NV,NV,NV,0,matched_exact,agree,True,45774,True,NaN


## Section 4 - Merge ISIC Metadata

In [5]:
meta_raw = pd.read_csv(METADATA_CSV, low_memory=False)
print(f"Metadata CSV: {len(meta_raw):,} rows, columns: {list(meta_raw.columns)}")

Metadata CSV: 25,331 rows, columns: ['image', 'age_approx', 'anatom_site_general', 'lesion_id', 'sex']


In [6]:
meta_id_col = meta_raw.columns[0]
print(f"Metadata image ID column: '{meta_id_col}'")

META_COLS_WANTED  = ["age_approx", "sex", "anatom_site_general", "lesion_id"]
meta_cols_present = [c for c in META_COLS_WANTED if c in meta_raw.columns]
meta_cols_missing = [c for c in META_COLS_WANTED if c not in meta_raw.columns]

if meta_cols_missing:
    print(f"  WARNING: columns not in metadata CSV: {meta_cols_missing}")
print(f"  Columns to merge: {meta_cols_present}")

meta = meta_raw[[meta_id_col] + meta_cols_present].copy()
meta = meta.rename(columns={meta_id_col: "meta_image_id"})
for col in META_COLS_WANTED:
    if col not in meta.columns:
        meta[col] = None

df = df.merge(meta, left_on="canonical_match_id", right_on="meta_image_id", how="left")
df["metadata_row_found"] = df["meta_image_id"].notna()
if "meta_image_id" in df.columns:
    df = df.drop(columns=["meta_image_id"])

n_meta_matched = int(df["metadata_row_found"].sum())
print(f"\nMetadata matched: {n_meta_matched:,} / {len(df):,} rows")
for col in META_COLS_WANTED:
    miss = int(df[col].isna().sum()) if col in df.columns else len(df)
    print(f"  {col} missing: {miss:,}")

Metadata image ID column: 'image'
  Columns to merge: ['age_approx', 'sex', 'anatom_site_general', 'lesion_id']

Metadata matched: 20,720 / 20,720 rows
  age_approx missing: 408
  sex missing: 358
  anatom_site_general missing: 2,293
  lesion_id missing: 1,886


## Section 5 - Compute SHA256 File Hashes

In [7]:
def sha256_file(path_str):
    try:
        h = hashlib.sha256()
        with open(path_str, "rb") as fh:
            for chunk in iter(lambda: fh.read(65536), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return ""

print("Computing SHA256 hashes...")
df["file_hash"] = df["full_path"].apply(sha256_file)
hash_ok   = int((df["file_hash"] != "").sum())
hash_fail = int((df["file_hash"] == "").sum())
print(f"  Hashed OK : {hash_ok:,}")
print(f"  Hash fail : {hash_fail:,}")

Computing SHA256 hashes...
  Hashed OK : 20,720
  Hash fail : 0


## Section 6 - Duplicate and Family Risk Analysis

In [8]:
# 6a - Hash group sizes
hash_valid_mask = df["file_hash"] != ""
hash_grp_sizes  = df[hash_valid_mask].groupby("file_hash")["full_path"].transform("count")
df["duplicate_hash_group_size"] = 0
df.loc[hash_valid_mask, "duplicate_hash_group_size"] = hash_grp_sizes.astype(int)

n_dup_hash_groups = int(
    df[hash_valid_mask & (df["duplicate_hash_group_size"] > 1)]["file_hash"].nunique()
)
n_dup_hash_rows = int((df["duplicate_hash_group_size"] > 1).sum())
print(f"Duplicate hash groups : {n_dup_hash_groups:,}  ({n_dup_hash_rows:,} rows)")

Duplicate hash groups : 28  (56 rows)


In [9]:
# 6b - Repeated canonical_match_id groups
cid_valid_mask = df["canonical_match_id"].notna() & (df["canonical_match_id"].astype(str) != "")
cid_grp_sizes  = df[cid_valid_mask].groupby("canonical_match_id")["full_path"].transform("count")
df["canonical_id_group_size"] = 1
df.loc[cid_valid_mask, "canonical_id_group_size"] = cid_grp_sizes.astype(int)

n_rep_canonical_groups = int(
    df[cid_valid_mask & (df["canonical_id_group_size"] > 1)]["canonical_match_id"].nunique()
)
n_rep_canonical_rows = int((df["canonical_id_group_size"] > 1).sum())
print(f"Repeated canonical_match_id groups: {n_rep_canonical_groups:,}  ({n_rep_canonical_rows:,} rows)")

Repeated canonical_match_id groups: 0  (0 rows)


In [10]:
# 6c - Lesion ID groups
lesion_col_present = "lesion_id" in df.columns and df["lesion_id"].notna().any()
if lesion_col_present:
    lesion_valid  = df["lesion_id"].notna() & (df["lesion_id"].astype(str) != "")
    les_grp_sizes = df[lesion_valid].groupby("lesion_id")["full_path"].transform("count")
    df["lesion_id_group_size"] = 1
    df.loc[lesion_valid, "lesion_id_group_size"] = les_grp_sizes.astype(int)
    n_lesion_groups = int(
        df[lesion_valid & (df["lesion_id_group_size"] > 1)]["lesion_id"].nunique()
    )
    n_lesion_rows = int((df["lesion_id_group_size"] > 1).sum())
    print(f"Repeated lesion_id groups: {n_lesion_groups:,}  ({n_lesion_rows:,} rows)")
else:
    df["lesion_id_group_size"] = 1
    n_lesion_groups = 0
    n_lesion_rows   = 0
    print("lesion_id column absent or all-null.")

Repeated lesion_id groups: 3,954  (13,044 rows)


In [11]:
# 6d - Suffix/downsampled count
n_suffix_variants = int((df["recognized_suffix"].fillna("") != "").sum())
print(f"Suffix/downsampled variants: {n_suffix_variants:,}")

Suffix/downsampled variants: 1,690


In [12]:
# 6e - Assign family_id  (lesion > hash > canonical_id, first match wins)
family_id_map = {}

if lesion_col_present:
    for lid, grp in df[df["lesion_id_group_size"] > 1].groupby("lesion_id"):
        fid = f"lesion_{lid}"
        for idx in grp.index:
            family_id_map[idx] = fid

for h, grp in df[df["duplicate_hash_group_size"] > 1].groupby("file_hash"):
    for idx in grp.index:
        if idx not in family_id_map:
            family_id_map[idx] = f"hash_{h[:12]}"

for cid, grp in df[df["canonical_id_group_size"] > 1].groupby("canonical_match_id"):
    for idx in grp.index:
        if idx not in family_id_map:
            family_id_map[idx] = f"cid_{cid}"

df["family_id"] = [family_id_map.get(i, "") for i in df.index]
n_fam_assigned  = int((df["family_id"] != "").sum())
print(f"family_id assigned to {n_fam_assigned:,} rows")

family_id assigned to 13,046 rows


In [13]:
# 6f - Duplicate risk flag and reason
def _dup_reasons(row):
    r = []
    if row.get("duplicate_hash_group_size", 1) > 1:
        r.append("duplicate_hash")
    if row.get("canonical_id_group_size", 1) > 1:
        r.append("repeated_canonical_id")
    if row.get("lesion_id_group_size", 1) > 1:
        r.append("shared_lesion_id")
    if str(row.get("recognized_suffix", "")) not in ("", "nan"):
        r.append("downsampled_variant")
    return r

dup_lists = df.apply(_dup_reasons, axis=1)
df["duplicate_risk_flag"]   = dup_lists.apply(lambda x: len(x) > 0)
df["duplicate_risk_reason"] = dup_lists.apply(lambda x: "; ".join(x))

n_dup_risk = int(df["duplicate_risk_flag"].sum())
print(f"Rows with duplicate_risk_flag=True: {n_dup_risk:,}")
print(df["duplicate_risk_reason"].value_counts().head(10).to_string())

Rows with duplicate_risk_flag=True: 14,701
duplicate_risk_reason
shared_lesion_id                         12957
                                          6019
downsampled_variant                       1655
duplicate_hash; shared_lesion_id            54
shared_lesion_id; downsampled_variant       33
duplicate_hash; downsampled_variant          2


## Section 7 - Image Quality Audit (Decode Eligible Images)

In [14]:
def _decode_cv2(path_str):
    try:
        img = cv2.imread(str(path_str), cv2.IMREAD_COLOR)
        if img is None:
            return None, "cv2_read_none"
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), None
    except Exception as e:
        return None, str(e)

def _decode_pil(path_str):
    try:
        with Image.open(path_str) as img:
            img.load()
            return np.array(img.convert("RGB")), None
    except Exception as e:
        return None, str(e)

def decode_image(path_str):
    if CV2_AVAILABLE:
        return _decode_cv2(path_str)
    elif PIL_AVAILABLE:
        return _decode_pil(path_str)
    return None, "no_decode_backend"

def _laplacian_var(gray_f32):
    if CV2_AVAILABLE:
        lap = cv2.Laplacian(gray_f32, cv2.CV_64F)
        return float(lap.var())
    dy = gray_f32[1:, :] - gray_f32[:-1, :]
    dx = gray_f32[:, 1:] - gray_f32[:, :-1]
    return float(np.var(dy) + np.var(dx))

def compute_quality(arr_rgb):
    h, w = arr_rgb.shape[:2]
    gray = arr_rgb.mean(axis=2).astype(np.float32)
    bm   = float(gray.mean())
    cs   = float(gray.std())
    dpf  = float((gray < 30).mean())
    slv  = _laplacian_var(gray)
    ar   = round(w / h, 4) if h > 0 else None
    return {
        "decoded_width":           w,
        "decoded_height":          h,
        "aspect_ratio":            ar,
        "brightness_mean":         round(bm, 3),
        "contrast_std":            round(cs, 3),
        "dark_pixel_fraction":     round(dpf, 5),
        "sharpness_laplacian_var": round(slv, 3),
    }

EMPTY_QUALITY = {
    "decoded_width": None, "decoded_height": None, "aspect_ratio": None,
    "brightness_mean": None, "contrast_std": None,
    "dark_pixel_fraction": None, "sharpness_laplacian_var": None,
}
print("Decode helpers defined.")

Decode helpers defined.


In [17]:
eligible_idx = df.index[elig_mask].tolist()
n_to_decode  = len(eligible_idx)
print(f"Decoding {n_to_decode:,} eligible images...")

quality_rows = {}
n_ok = n_fail = 0

for i, idx in enumerate(eligible_idx):
    path_str = df.at[idx, "full_path"]
    arr, err = decode_image(path_str)
    if arr is not None:
        m = compute_quality(arr)
        m["image_decode_success"] = True
        m["decode_error"]         = ""
        n_ok += 1
    else:
        m = dict(EMPTY_QUALITY)
        m["image_decode_success"] = False
        m["decode_error"]         = err or "unknown"
        n_fail += 1
    quality_rows[idx] = m

    if (i + 1) % 1000 == 0 or (i + 1) == n_to_decode:
        print(f"  {i + 1:,}/{n_to_decode:,}  ok={n_ok:,}  fail={n_fail:,}")

n_decode_fail = n_fail
print(f"\nDecode complete: {n_ok:,} ok, {n_fail:,} failed.")

Decoding 20,720 eligible images...
  1,000/20,720  ok=1,000  fail=0
  2,000/20,720  ok=2,000  fail=0
  3,000/20,720  ok=3,000  fail=0
  4,000/20,720  ok=4,000  fail=0
  5,000/20,720  ok=5,000  fail=0
  6,000/20,720  ok=6,000  fail=0
  7,000/20,720  ok=7,000  fail=0
  8,000/20,720  ok=8,000  fail=0
  9,000/20,720  ok=9,000  fail=0
  10,000/20,720  ok=10,000  fail=0
  11,000/20,720  ok=11,000  fail=0
  12,000/20,720  ok=12,000  fail=0
  13,000/20,720  ok=13,000  fail=0
  14,000/20,720  ok=14,000  fail=0
  15,000/20,720  ok=15,000  fail=0
  16,000/20,720  ok=16,000  fail=0
  17,000/20,720  ok=17,000  fail=0
  18,000/20,720  ok=18,000  fail=0
  19,000/20,720  ok=19,000  fail=0
  20,000/20,720  ok=20,000  fail=0
  20,720/20,720  ok=20,720  fail=0

Decode complete: 20,720 ok, 0 failed.


In [18]:
QUALITY_COLS = [
    "image_decode_success", "decode_error",
    "decoded_width", "decoded_height", "aspect_ratio",
    "brightness_mean", "contrast_std",
    "dark_pixel_fraction", "sharpness_laplacian_var",
]

quality_df = pd.DataFrame.from_dict(quality_rows, orient="index")

for col in QUALITY_COLS:
    df[col] = quality_df[col] if col in quality_df.columns else None

for col in QUALITY_COLS:
    if col == "image_decode_success":
        df[col] = df[col].where(elig_mask, other=False)
    else:
        df[col] = df[col].where(elig_mask, other=None)

print("Quality columns merged.")
desc_cols = ["decoded_width","decoded_height","brightness_mean","contrast_std","sharpness_laplacian_var"]
print(df[desc_cols].describe().T[["count","mean","min","max"]].to_string())

Quality columns merged.
                           count        mean      min       max
decoded_width            20720.0  851.705550  576.000  1024.000
decoded_height           20720.0  754.949276  450.000  1024.000
brightness_mean          20720.0  147.344617   32.657   238.921
contrast_std             20720.0   37.118369    5.678   105.796
sharpness_laplacian_var  20720.0   26.877693    1.376  1435.170


## Section 8 - Quality Flag Assignment

In [19]:
def _quality_reasons(row):
    if not row.get("eligible_after_reconciliation", False):
        return []
    if not row.get("image_decode_success", False):
        return ["decode_failure"]
    r   = []
    w   = row.get("decoded_width")
    h   = row.get("decoded_height")
    ar  = row.get("aspect_ratio")
    bm  = row.get("brightness_mean")
    cs  = row.get("contrast_std")
    slv = row.get("sharpness_laplacian_var")
    if w is not None and h is not None and min(w, h) < MIN_DIM_PX:
        r.append("very_small_dimensions")
    if ar is not None and (ar > EXTREME_AR_HIGH or ar < EXTREME_AR_LOW):
        r.append("extreme_aspect_ratio")
    if bm is not None:
        if bm < DARK_THRESH:
            r.append("too_dark")
        elif bm > BRIGHT_THRESH:
            r.append("too_bright")
    if cs is not None and cs < LOW_CONTRAST_THRESH:
        r.append("very_low_contrast")
    if slv is not None and slv < LOW_SHARP_THRESH:
        r.append("very_low_sharpness")
    return r

q_lists = df.apply(_quality_reasons, axis=1)
df["quality_flag"]         = q_lists.apply(lambda x: len(x) > 0)
df["quality_flag_reasons"] = q_lists.apply(lambda x: "; ".join(x))

n_quality_flagged = int(df["quality_flag"].sum())
print(f"Quality-flagged rows: {n_quality_flagged:,}")
all_qr = []
for r in q_lists:
    all_qr.extend(r)
if all_qr:
    for reason, cnt in sorted(Counter(all_qr).items(), key=lambda x: -x[1]):
        print(f"  {reason:<30}: {cnt:,}")
else:
    print("  (none flagged)")

Quality-flagged rows: 1,879
  very_low_sharpness            : 1,879


## Section 9 - Build Audited Manifest Columns

In [20]:
df["needs_manual_review"] = (
    (df["eligible_after_reconciliation"] == True)
    & (df["duplicate_risk_flag"] | df["quality_flag"])
)

def _audit_issue(row):
    parts = []
    if row.get("duplicate_risk_flag", False) and row.get("duplicate_risk_reason", ""):
        parts.append(str(row["duplicate_risk_reason"]))
    if row.get("quality_flag", False) and row.get("quality_flag_reasons", ""):
        parts.append(str(row["quality_flag_reasons"]))
    return "; ".join(parts)

df["audit_issue_reason"] = df.apply(_audit_issue, axis=1)

df["eligible_after_audit_candidate"] = (
    (df["eligible_after_reconciliation"] == True)
    & (~df["needs_manual_review"])
)

print(f"needs_manual_review=True         : {int(df['needs_manual_review'].sum()):,}")
print(f"eligible_after_audit_candidate   : {int(df['eligible_after_audit_candidate'].sum()):,}")

needs_manual_review=True         : 14,809
eligible_after_audit_candidate   : 5,911


## Section 10 - Build Manual Review Queue

In [21]:
def _classify_review(row):
    types, reasons = [], []
    if row.get("duplicate_risk_flag", False):
        types.append("duplicate_family")
        reasons.append(str(row.get("duplicate_risk_reason", "")))
    if row.get("quality_flag", False):
        types.append("quality_flag")
        reasons.append(str(row.get("quality_flag_reasons", "")))
    rtype   = "; ".join(types)
    rreason = "; ".join(r for r in reasons if r)
    if "decode_failure" in rreason:
        action = "exclude_if_unrecoverable"
    elif "duplicate_hash" in rreason:
        action = "keep_one_exclude_rest"
    elif "shared_lesion_id" in rreason:
        action = "review_keep_or_group_split"
    elif "very_small_dimensions" in rreason or "extreme_aspect_ratio" in rreason:
        action = "exclude_if_corrupt"
    elif any(q in rreason for q in ["too_dark","too_bright","very_low_contrast","very_low_sharpness"]):
        action = "review_borderline"
    else:
        action = "review_manual"
    return rtype, rreason, action

review_rows = []
for idx, row in df[df["needs_manual_review"] == True].iterrows():
    rtype, rreason, raction = _classify_review(row)
    review_rows.append({
        "full_path":                 row["full_path"],
        "canonical_match_id":        row.get("canonical_match_id", ""),
        "final_authoritative_label": row.get("final_authoritative_label", ""),
        "review_type":               rtype,
        "review_reason":             rreason,
        "suggested_action":          raction,
        "manual_decision":           "",
        "manual_reason":             "",
        "reviewer_notes":            "",
    })

review_queue_df = pd.DataFrame(review_rows)
print(f"Review queue rows: {len(review_queue_df):,}")
if not review_queue_df.empty:
    print(review_queue_df["review_type"].value_counts().to_string())

Review queue rows: 14,809
review_type
duplicate_family                  12930
duplicate_family; quality_flag     1771
quality_flag                        108


## Section 11 - Build Audit Summary

In [22]:
elig_class_counts = (
    df[df["eligible_after_reconciliation"] == True]["final_authoritative_label"]
    .value_counts().to_dict()
)
rq_class_counts = (
    review_queue_df["final_authoritative_label"].value_counts().to_dict()
    if not review_queue_df.empty else {}
)

summary_rows_list = [
    {"metric": "total_rows_loaded",                 "value": len(df)},
    {"metric": "reconciliation_eligible_rows",      "value": int(elig_mask.sum())},
    {"metric": "metadata_matched_rows",             "value": int(df["metadata_row_found"].sum())},
    {"metric": "missing_age_approx",                "value": int(df["age_approx"].isna().sum()) if "age_approx" in df.columns else len(df)},
    {"metric": "missing_sex",                       "value": int(df["sex"].isna().sum()) if "sex" in df.columns else len(df)},
    {"metric": "missing_anatom_site_general",       "value": int(df["anatom_site_general"].isna().sum()) if "anatom_site_general" in df.columns else len(df)},
    {"metric": "missing_lesion_id",                 "value": int(df["lesion_id"].isna().sum()) if "lesion_id" in df.columns else len(df)},
    {"metric": "duplicate_hash_group_count",        "value": n_dup_hash_groups},
    {"metric": "repeated_canonical_id_group_count", "value": n_rep_canonical_groups},
    {"metric": "repeated_lesion_id_group_count",    "value": n_lesion_groups},
    {"metric": "suffix_downsampled_count",          "value": n_suffix_variants},
    {"metric": "decode_failure_count",              "value": n_decode_fail},
    {"metric": "quality_candidate_count",           "value": n_quality_flagged},
    {"metric": "review_queue_row_count",            "value": len(review_queue_df)},
]
for label in ["NV", "MEL", "BCC"]:
    summary_rows_list.append({"metric": f"eligible_class_{label}", "value": int(elig_class_counts.get(label, 0))})
for label in ["NV", "MEL", "BCC"]:
    summary_rows_list.append({"metric": f"review_queue_class_{label}", "value": int(rq_class_counts.get(label, 0))})

summary_df = pd.DataFrame(summary_rows_list)
print(summary_df.to_string(index=False))

                           metric  value
                total_rows_loaded  20720
     reconciliation_eligible_rows  20720
            metadata_matched_rows  20720
               missing_age_approx    408
                      missing_sex    358
      missing_anatom_site_general   2293
                missing_lesion_id   1886
       duplicate_hash_group_count     28
repeated_canonical_id_group_count      0
   repeated_lesion_id_group_count   3954
         suffix_downsampled_count   1690
             decode_failure_count      0
          quality_candidate_count   1879
           review_queue_row_count  14809
                eligible_class_NV  12875
               eligible_class_MEL   4522
               eligible_class_BCC   3323
            review_queue_class_NV   7727
           review_queue_class_MEL   4092
           review_queue_class_BCC   2990


## Section 12 - Save Outputs

In [23]:
audited_manifest_path = OUTPUT_AUDIT_DIR / "02_audited_manifest.csv"
df.to_csv(audited_manifest_path, index=False)
print(f"Saved audited manifest ({len(df):,} rows) -> {audited_manifest_path}")

Saved audited manifest (20,720 rows) -> C:\SKIN CANCER v2\pipe output\02_audit_review\02_audited_manifest.csv


In [24]:
audit_summary_path = OUTPUT_AUDIT_DIR / "02_audit_summary.csv"
summary_df.to_csv(audit_summary_path, index=False)
print(f"Saved audit summary ({len(summary_df)} rows) -> {audit_summary_path}")

Saved audit summary (20 rows) -> C:\SKIN CANCER v2\pipe output\02_audit_review\02_audit_summary.csv


In [25]:
review_queue_path = OUTPUT_AUDIT_DIR / "02_review_queue.csv"
if not review_queue_df.empty:
    review_queue_df.to_csv(review_queue_path, index=False)
else:
    pd.DataFrame(columns=[
        "full_path", "canonical_match_id", "final_authoritative_label",
        "review_type", "review_reason", "suggested_action",
        "manual_decision", "manual_reason", "reviewer_notes",
    ]).to_csv(review_queue_path, index=False)
print(f"Saved review queue ({len(review_queue_df):,} rows) -> {review_queue_path}")

Saved review queue (14,809 rows) -> C:\SKIN CANCER v2\pipe output\02_audit_review\02_review_queue.csv


## Section 13 - Output File Verification

In [26]:
output_files = [
    OUTPUT_AUDIT_DIR / "02_audited_manifest.csv",
    OUTPUT_AUDIT_DIR / "02_audit_summary.csv",
    OUTPUT_AUDIT_DIR / "02_review_queue.csv",
]
print("Output file verification:")
all_ok = True
for p in output_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<42} {size:>12,} bytes")
    if not exists:
        all_ok = False
print("\nAll files present." if all_ok else "\nWARNING: missing files.")

Output file verification:
  [OK] 02_audited_manifest.csv                       8,000,167 bytes
  [OK] 02_audit_summary.csv                                584 bytes
  [OK] 02_review_queue.csv                           1,905,457 bytes

All files present.


## Section 14 - Final Summary (Copy-Paste Ready)

In [27]:
print("=" * 64)
print("  AUDIT NOTEBOOK -- FINAL SUMMARY")
print("=" * 64)

print(f"\n1.  Total rows loaded                    : {len(df):,}")
print(f"2.  Eligible from reconciliation         : {int(elig_mask.sum()):,}")

print(f"\n3.  Metadata:")
print(f"    Matched rows                         : {int(df['metadata_row_found'].sum()):,}")
for col in ["age_approx", "sex", "anatom_site_general", "lesion_id"]:
    miss = int(df[col].isna().sum()) if col in df.columns else len(df)
    print(f"    Missing {col:<26} : {miss:,}")

print(f"\n4.  Duplicate / family risk:")
print(f"    Rows with duplicate_risk_flag=True   : {int(df['duplicate_risk_flag'].sum()):,}")
print(f"    Duplicate hash groups                : {n_dup_hash_groups:,}")
print(f"    Repeated canonical_id groups         : {n_rep_canonical_groups:,}")
print(f"    Repeated lesion_id groups            : {n_lesion_groups:,}")
print(f"    Suffix/downsampled variants          : {n_suffix_variants:,}")

print(f"\n5.  Decode failures                      : {n_decode_fail:,}")

print(f"\n6.  Quality flag counts by reason:")
all_qr2 = []
for r in df[df["quality_flag"] == True]["quality_flag_reasons"]:
    all_qr2.extend([x for x in str(r).split("; ") if x])
if all_qr2:
    for reason, cnt in sorted(Counter(all_qr2).items(), key=lambda x: -x[1]):
        print(f"    {reason:<34}: {cnt:,}")
else:
    print("    (none)")

print(f"\n7.  Review queue row count               : {len(review_queue_df):,}")

print(f"\n8.  Review queue by review_type:")
if not review_queue_df.empty:
    tc = Counter()
    for t in review_queue_df["review_type"]:
        for part in t.split("; "):
            if part:
                tc[part] += 1
    for rtype, cnt in sorted(tc.items(), key=lambda x: -x[1]):
        print(f"    {rtype:<34}: {cnt:,}")
else:
    print("    (queue empty)")

print(f"\n9.  Class counts (reconciliation-eligible):")
for label in ["NV", "MEL", "BCC"]:
    print(f"    {label:<5}: {int(elig_class_counts.get(label, 0)):,}")

print(f"\n10. Class counts (review queue):")
for label in ["NV", "MEL", "BCC"]:
    print(f"    {label:<5}: {int(rq_class_counts.get(label, 0)):,}")

print(f"\n11. Audited manifest columns ({len(df.columns)}):")
print("    " + ", ".join(df.columns.tolist()))

print(f"\n12. Output file verification:")
for p in output_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"    [{status}] {p.name:<42} {size:>12,} bytes")

print("=" * 64)

  AUDIT NOTEBOOK -- FINAL SUMMARY

1.  Total rows loaded                    : 20,720
2.  Eligible from reconciliation         : 20,720

3.  Metadata:
    Matched rows                         : 20,720
    Missing age_approx                 : 408
    Missing sex                        : 358
    Missing anatom_site_general        : 2,293
    Missing lesion_id                  : 1,886

4.  Duplicate / family risk:
    Rows with duplicate_risk_flag=True   : 14,701
    Duplicate hash groups                : 28
    Repeated canonical_id groups         : 0
    Repeated lesion_id groups            : 3,954
    Suffix/downsampled variants          : 1,690

5.  Decode failures                      : 0

6.  Quality flag counts by reason:
    very_low_sharpness                : 1,879

7.  Review queue row count               : 14,809

8.  Review queue by review_type:
    duplicate_family                  : 14,701
    quality_flag                      : 1,879

9.  Class counts (reconciliation-eligibl

In [28]:
print("\n13. Audited manifest head(3):")
display(df.head(3))


13. Audited manifest head(3):


,full_path,folder_name,file_name_with_extension,file_stem_raw,extension,recognized_suffix,base_id_candidate,canonical_match_id,folder_label,gt_label,...,aspect_ratio,brightness_mean,contrast_std,dark_pixel_fraction,sharpness_laplacian_var,quality_flag,quality_flag_reasons,needs_manual_review,audit_issue_reason,eligible_after_audit_candidate
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,ISIC_0000000.jpg,ISIC_0000000,.jpg,NaN,ISIC_0000000,ISIC_0000000,NV,NV,...,1.3325,150.324,68.582,0.00369,14.270,False,,False,,True
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,ISIC_0000001.jpg,ISIC_0000001,.jpg,NaN,ISIC_0000001,ISIC_0000001,NV,NV,...,1.3325,164.697,31.539,0.00811,34.892,False,,False,,True
2,C:\SKIN CANCER v2\DS\NV\ISIC_0000003.jpg,NV,ISIC_0000003.jpg,ISIC_0000003,.jpg,NaN,ISIC_0000003,ISIC_0000003,NV,NV,...,1.3325,178.489,59.127,0.00001,13.910,False,,False,,True


In [29]:
print("\n14. Review queue head(10):")
if not review_queue_df.empty:
    display(review_queue_df.head(10))
else:
    print("  (review queue is empty)")


14. Review queue head(10):


,full_path,canonical_match_id,final_authoritative_label,review_type,review_reason,suggested_action,manual_decision,manual_reason,reviewer_notes
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000017_downsampl...,ISIC_0000017_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000018_downsampl...,ISIC_0000018_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,
2,C:\SKIN CANCER v2\DS\NV\ISIC_0000019_downsampl...,ISIC_0000019_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,
3,C:\SKIN CANCER v2\DS\NV\ISIC_0000020_downsampl...,ISIC_0000020_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,
4,C:\SKIN CANCER v2\DS\NV\ISIC_0000021_downsampl...,ISIC_0000021_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,
5,C:\SKIN CANCER v2\DS\NV\ISIC_0000023_downsampl...,ISIC_0000023_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,
6,C:\SKIN CANCER v2\DS\NV\ISIC_0000024_downsampl...,ISIC_0000024_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,
7,C:\SKIN CANCER v2\DS\NV\ISIC_0000025_downsampl...,ISIC_0000025_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,
8,C:\SKIN CANCER v2\DS\NV\ISIC_0000027_downsampl...,ISIC_0000027_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,
9,C:\SKIN CANCER v2\DS\NV\ISIC_0000028_downsampl...,ISIC_0000028_downsampled,NV,duplicate_family,downsampled_variant,review_manual,,,


## Section 15 - Completion Summary

**Section 01 - Metadata Merge, Duplicate Audit, and Image Quality Audit is complete.**

What was accomplished in this notebook:
- Reconciled manifest loaded from `01_reconciliation/01_reconciled_manifest.csv`.
- ISIC metadata merged by `canonical_match_id`; `age_approx`, `sex`, `anatom_site_general`, `lesion_id`, `metadata_row_found` added.
- SHA256 file hashes computed for all files.
- Duplicate/family risk assessed: hash groups, repeated canonical IDs, lesion ID groups, downsampled suffix variants detected.
- `family_id` assigned to all images sharing a hash, lesion ID, or canonical ID (lesion priority).
- All reconciliation-eligible images decoded; quality metrics computed.
- Quality candidates flagged conservatively: decode failure, very small dims, extreme aspect ratio, too dark, too bright, very low contrast, very low sharpness.
- `needs_manual_review` set for eligible rows with any duplicate or quality flag.
- Combined review queue built with `manual_decision` column blank for reviewer input.
- All three required output files saved. No rows dropped.

**What was deliberately deferred:**
- Acting on `manual_decision` values (keep / exclude / uncertain)
- Final row exclusions based on audit decisions
- Preprocessing, augmentation, and train/val/test splitting

**Next notebook:** `02_exclusion_and_final_manifest.ipynb`
Load `02_review_queue.csv` after `manual_decision` has been populated, apply exclusions, and produce the final clean dataset manifest ready for splitting and training.